In [11]:
import os
from glob import glob

import numpy as np
import xarray as xr



In [17]:
# =====================
# Settings
# =====================
year_start = 1990
year_end = 2025

threshold_percentile = 20
persistence_threshold = 3  # consecutive days

u_dir = "/g/data/nf33/WindDroughts_Group3/ERA5/100u"
v_dir = "/g/data/nf33/WindDroughts_Group3/ERA5/100v"

threshold_file = "/g/data/nf33/WindDroughts_Group3/ls3248/ERA5_WS_P20_1991_2020_AUS.nc"
out_dir = "/g/data/nf33/WindDroughts_Group3/ls3248/wind_drought_ERA5"

# =====================
# Load threshold field
# =====================
p20 = xr.open_dataset(threshold_file)["ws_p20"].load()

# Use threshold grid to define the region
lat_min = float(p20.latitude.min())
lat_max = float(p20.latitude.max())
lon_min = float(p20.longitude.min())
lon_max = float(p20.longitude.max())

# ERA5 latitude is descending
lat_slice = slice(lat_max, lat_min)
lon_slice = slice(lon_min, lon_max)

print("Loaded threshold field:", p20.shape, flush=True)




Loaded threshold field: (141, 201)


In [18]:
# =====================
# Helper: load one year's monthly files
# =====================
def load_year_data(file_list, var_name):
    arrays = []
    for fp in file_list:
        da = xr.open_dataset(fp)[var_name]
        da = da.sel(latitude=lat_slice, longitude=lon_slice).load()
        arrays.append(da)
    return xr.concat(arrays, dim="time")


# =====================
# Helper: detect persistent drought along time only
# =====================
def drought_filter_1d(mask_1d, min_len=3):
    """
    Input:
        mask_1d: 1D boolean array along time
    Output:
        1D int8 array, 1 = drought event day, 0 = non-drought
    """
    mask_1d = np.asarray(mask_1d, dtype=bool)
    out = np.zeros(mask_1d.shape, dtype=np.int8)

    start = None
    for i, flag in enumerate(mask_1d):
        if flag:
            if start is None:
                start = i
        else:
            if start is not None:
                if i - start >= min_len:
                    out[start:i] = 1
                start = None

    if start is not None and (len(mask_1d) - start) >= min_len:
        out[start:] = 1

    return out




In [19]:
# =====================
# Main yearly loop
# =====================
for year in range(year_start, year_end + 1):
    print(f"\n===== Processing {year} =====", flush=True)

    u_files = sorted(glob(f"{u_dir}/100u_era5_oper_sfc_daily_{year}*.nc"))
    v_files = sorted(glob(f"{v_dir}/100v_era5_oper_sfc_daily_{year}*.nc"))

    print(f"Found {len(u_files)} u files and {len(v_files)} v files.", flush=True)

    # Load one year at a time
    ds_u = load_year_data(u_files, "u100")
    ds_v = load_year_data(v_files, "v100")

    # Daily wind speed
    ws = np.hypot(ds_u, ds_v).astype("float32").rename("ws")
    print("wind speed calculated")

    # Make sure grid matches threshold field exactly
    ws = ws.sel(latitude=p20.latitude, longitude=p20.longitude)
    print("region selected")

    # Candidate drought days: WS < P20
    candidate = (ws < p20)

    # Keep only persistent drought events with length >= persistence_threshold
    wind_drought = xr.apply_ufunc(
        drought_filter_1d,
        candidate,
        kwargs={"min_len": persistence_threshold},
        input_core_dims=[["time"]],
        output_core_dims=[["time"]],
        vectorize=True,
        output_dtypes=[np.int8],
    )

    wind_drought = wind_drought.transpose("time", "latitude", "longitude")
    wind_drought.name = "wind_drought"

    wind_drought.attrs["long_name"] = (
        f"Wind drought mask: 1 = WS < P{threshold_percentile} "
        f"for at least {persistence_threshold} consecutive days"
    )
    wind_drought.attrs["definition"] = "1 = wind drought event day, 0 = non-drought"
    wind_drought.attrs["threshold_percentile"] = threshold_percentile
    wind_drought.attrs["persistence_threshold_days"] = persistence_threshold
    wind_drought.attrs["units"] = "1"

    outfile = os.path.join(
        out_dir,
        f"wind_drought_P{threshold_percentile}_{persistence_threshold}day_{year}.nc"
    )

    wind_drought.astype("int8").to_netcdf(outfile)
    print(f"Saved: {outfile}", flush=True)

    # Free memory
    del ds_u, ds_v, ws, candidate, wind_drought

print("\nAll years done.", flush=True)


===== Processing 1990 =====
Found 12 u files and 12 v files.
wind speed calculated
region selected
Saved: /g/data/nf33/WindDroughts_Group3/ls3248/wind_drought_ERA5/wind_drought_P20_3day_1990.nc

===== Processing 1991 =====
Found 12 u files and 12 v files.
wind speed calculated
region selected
Saved: /g/data/nf33/WindDroughts_Group3/ls3248/wind_drought_ERA5/wind_drought_P20_3day_1991.nc

===== Processing 1992 =====
Found 12 u files and 12 v files.
wind speed calculated
region selected
Saved: /g/data/nf33/WindDroughts_Group3/ls3248/wind_drought_ERA5/wind_drought_P20_3day_1992.nc

===== Processing 1993 =====
Found 12 u files and 12 v files.
wind speed calculated
region selected
Saved: /g/data/nf33/WindDroughts_Group3/ls3248/wind_drought_ERA5/wind_drought_P20_3day_1993.nc

===== Processing 1994 =====
Found 12 u files and 12 v files.
wind speed calculated
region selected
Saved: /g/data/nf33/WindDroughts_Group3/ls3248/wind_drought_ERA5/wind_drought_P20_3day_1994.nc

===== Processing 1995 ==